# 03a — Exp 1: Post-hoc Logit Adjustment

**Project:** UREP 32-0210-250078 | Crack Classification

**Method:** At inference, subtract τ · log(π_y) from each class logit, where π_y is the training frequency of class y (Menon et al. 2021).

**Why:** Free win — no retraining required. Tells us immediately how much of the shear-recall problem is classifier bias vs label noise.

**Setup:** Sweep τ ∈ {0.5, 1.0, 1.5, 2.0} on validation set. Pick τ that maximizes macro-F1.

**Expected:** Shear recall +5–10 points, macro-F1 +1–3 points.

## Setup

In [ ]:
import sys
sys.path.insert(0, "..")

import os
import json
import numpy as np
import pandas as pd
import torch

import config
from src.device import print_device_summary, get_device, set_seed
from src.evaluation import evaluate_predictions
from src.model_cbam_hierarchical import InceptionV3CBAMHierarchical
from src.hierarchical import (
    STAGE1_CLASSES, STAGE2_CLASSES, STAGE3_CLASSES,
    get_hierarchical_dataloaders,
    evaluate_hierarchical_model,
    hierarchical_pr_f1, per_stage_confusion_matrices, error_attribution,
)
from src.losses import (
    compute_class_frequencies,
    apply_logit_adjustment,
    evaluate_hierarchical_with_logit_adj,
    HierarchicalCrackDataset,
)
from src.augmentation import get_val_test_transforms

set_seed(config.RANDOM_SEED)

OUTPUT_DIR = os.path.join(config.OUTPUT_DIR, "exp1_logit_adj")
os.makedirs(os.path.join(OUTPUT_DIR, "plots"), exist_ok=True)

device_config = print_device_summary()
device = get_device()
BATCH = device_config["batch_sizes"][1]
NUM_WORKERS = device_config["num_workers"]

print(f"\nExperiment 1: Post-hoc Logit Adjustment")
print(f"Device: {device}")

## Load trained hierarchical CBAM model

In [ ]:
MODEL_PATH = os.path.join(config.OUTPUT_DIR, "cbam_hier_multix2", "models", "best_model.pt")

model = InceptionV3CBAMHierarchical().to(device)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device, weights_only=True))
model.eval()
print(f"Loaded model from {MODEL_PATH}")

## Compute per-stage class frequencies from training set

In [ ]:
train_ds = HierarchicalCrackDataset(
    config.SPLIT_DIR, "train",
    transform=get_val_test_transforms(config.IMG_SIZE, "imagenet"),
)

freqs = compute_class_frequencies(train_ds)

print("Per-stage class frequencies:")
for stage, names in [("stage1", STAGE1_CLASSES), ("stage2", STAGE2_CLASSES), ("stage3", STAGE3_CLASSES)]:
    f = freqs[stage]
    print(f"  {stage}: " + "  ".join(f"{n}={v:.4f}" for n, v in zip(names, f)))

# Precompute log-frequencies as tensors
log_freqs = {
    k: torch.tensor(np.log(v + 1e-8), dtype=torch.float32)
    for k, v in freqs.items()
}

del train_ds

## Build val and test loaders

In [ ]:
_, val_loader, test_loader = get_hierarchical_dataloaders(
    split_dir=config.SPLIT_DIR,
    batch_size=BATCH,
    img_size=config.IMG_SIZE,
    num_workers=NUM_WORKERS,
    sampler_kind="stage1",
)

## Baseline (no adjustment) on validation set

In [ ]:
from sklearn.metrics import f1_score, recall_score, classification_report

baseline_results = evaluate_hierarchical_model(model, val_loader, device, t1=0.5, t2=0.5)

y_true_idx = np.array([config.CLASS_NAMES.index(c) for c in baseline_results["y_true_flat"]])
y_pred_idx = np.array([config.CLASS_NAMES.index(c) for c in baseline_results["y_pred_flat"]])

baseline_f1 = f1_score(y_true_idx, y_pred_idx, average="macro")
shear_idx = config.CLASS_NAMES.index("shear")
shear_mask = y_true_idx == shear_idx
baseline_shear_recall = recall_score(y_true_idx, y_pred_idx, average=None)[shear_idx]

print(f"Baseline (τ=0): macro-F1={baseline_f1:.4f}  shear_recall={baseline_shear_recall:.4f}")

## Sweep τ on validation set

Try shared τ across all 3 heads first.

In [ ]:
tau_values = [0.0, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0]
sweep_results = []

for tau in tau_values:
    results = evaluate_hierarchical_with_logit_adj(
        model, val_loader, device, log_freqs,
        tau=tau, t1=0.5, t2=0.5,
    )

    yt = np.array([config.CLASS_NAMES.index(c) for c in results["y_true_flat"]])
    yp = np.array([config.CLASS_NAMES.index(c) for c in results["y_pred_flat"]])

    macro_f1 = f1_score(yt, yp, average="macro")
    per_class_recall = recall_score(yt, yp, average=None)
    per_class_f1 = f1_score(yt, yp, average=None)
    shear_rec = per_class_recall[shear_idx]
    shear_f1 = per_class_f1[shear_idx]

    sweep_results.append({
        "tau": tau,
        "macro_f1": macro_f1,
        "shear_recall": shear_rec,
        "shear_f1": shear_f1,
    })
    print(f"  τ={tau:.2f}  macro-F1={macro_f1:.4f}  shear_recall={shear_rec:.4f}  shear_f1={shear_f1:.4f}")

df_sweep = pd.DataFrame(sweep_results)
print("\n", df_sweep.to_string(index=False))

## Select best τ and evaluate on test set

In [ ]:
best_row = df_sweep.loc[df_sweep["macro_f1"].idxmax()]
best_tau = best_row["tau"]
print(f"Best τ = {best_tau:.2f} (val macro-F1 = {best_row['macro_f1']:.4f})")

# Evaluate on test set with best τ
test_results = evaluate_hierarchical_with_logit_adj(
    model, test_loader, device, log_freqs,
    tau=best_tau, t1=0.5, t2=0.5,
)

y_true_idx = np.array([config.CLASS_NAMES.index(c) for c in test_results["y_true_flat"]])
y_pred_idx = np.array([config.CLASS_NAMES.index(c) for c in test_results["y_pred_flat"]])

metrics = evaluate_predictions(
    y_true_idx, y_pred_idx,
    output_dir=OUTPUT_DIR, model_name="exp1_logit_adj",
)

## Per-stage confusion matrices, hierarchical metrics, error attribution

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

stage_cms = per_stage_confusion_matrices(test_results["y_true_paths"], test_results["y_pred_paths"])
h_metrics = hierarchical_pr_f1(test_results["y_true_paths"], test_results["y_pred_paths"])
err_attr  = error_attribution(test_results["y_true_paths"], test_results["y_pred_paths"])

print("Hierarchical precision/recall/F1:")
for k, v in h_metrics.items():
    print(f"  {k}: {v:.4f}")

print(f"\nError attribution:")
print(f"  Total:        {err_attr['total']}")
print(f"  Correct:      {err_attr['correct']}")
print(f"  Stage1 errs:  {err_attr['errors_by_stage']['stage1']}")
print(f"  Stage2 errs:  {err_attr['errors_by_stage']['stage2']}")
print(f"  Stage3 errs:  {err_attr['errors_by_stage']['stage3']}")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, key in zip(axes, ["stage1", "stage2", "stage3"]):
    if key not in stage_cms:
        ax.set_visible(False); continue
    info = stage_cms[key]
    sns.heatmap(info["cm"], annot=True, fmt="d", cmap="Blues",
                xticklabels=info["classes"], yticklabels=info["classes"], ax=ax)
    ax.set_title(f"{key}  ({info['cm'].sum()} samples)")
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "plots", "per_stage_confusion_matrices.png"),
            dpi=150, bbox_inches="tight")
plt.show()

## Save all metrics

In [ ]:
with open(os.path.join(OUTPUT_DIR, "exp1_results.json"), "w") as f:
    json.dump({
        "best_tau": float(best_tau),
        "tau_sweep": sweep_results,
        "hierarchical": h_metrics,
        "error_attribution": err_attr,
        "flat": {
            "accuracy": metrics["accuracy"],
            "f1_macro": metrics["f1_macro"],
            "f1_weighted": metrics["f1_weighted"],
        },
    }, f, indent=2)
print(f"\nSaved results to {OUTPUT_DIR}/exp1_results.json")